In [ ]:
import cv2
import numpy as np
import re
from collections import deque
from ultralytics import YOLO
from paddleocr import PaddleOCR


In [ ]:
model = YOLO(
    r"D:\Automatic ANPR\Self_Code\runs\detect\lp_yolov8s_stage2_final3\weights\best.pt"
)


In [ ]:
from paddleocr import PaddleOCR

ocr_engine = PaddleOCR(
    use_angle_cls=True,
    lang='en',
    det=True,
    rec=True,
    use_gpu=False  # change if GPU
)


[2026/02/12 09:55:24] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\100ra/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\100ra/.paddleocr/whl\\rec\\en\\en_PP-OCRv3_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_length=25, re

In [ ]:
def preprocess_plate_for_paddle(plate):
    # add margin safety
    h, w = plate.shape[:2]
    
    # upscale small plates
    scale = 2.0
    plate = cv2.resize(plate, (int(w * scale), int(h * scale)))
    
    # convert to gray
    gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)
    
    # mild CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))
    gray = clahe.apply(gray)
    
    return gray


In [ ]:
def ocr_plate_paddle(plate_img):
    result = ocr_engine.ocr(plate_img, cls=True)
    
    if not result or not result[0]:
        return ""
    
    texts = []
    
    for line in result[0]:
        text = line[1][0]
        confidence = line[1][1]
        
        if confidence > 0.6:
            texts.append(text)
    
    if not texts:
        return ""
    
    combined = "".join(texts)
    combined = combined.upper()
    combined = re.sub(r'[^A-Z0-9]', '', combined)
    
    return combined


In [ ]:
def normalize_plate_text(text):
    text = text.upper()
    text = re.sub(r'[^A-Z0-9]', '', text)
    
    # common confusion corrections
    text = text.replace("O", "0")
    text = text.replace("I", "1")
    text = text.replace("Z", "2")
    text = text.replace("S", "5")
    
    return text


In [ ]:
def validate_indian_plate(text):
    pattern1 = r'^[A-Z]{2}[0-9]{2}[A-Z]{1}[0-9]{4}$'
    pattern2 = r'^[A-Z]{2}[0-9]{2}[A-Z]{2}[0-9]{4}$'
    
    return bool(re.match(pattern1, text) or re.match(pattern2, text))


## Image Inference Pipeline (First Test)

In [11]:
img_path = r"D:\Automatic ANPR\Self_Code\dataset_original\images\val\WB8.jpg"
img = cv2.imread(img_path)

results = model(img, conf=0.4)

for r in results:
    for box in r.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        
        # add small padding
        pad = 5
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(img.shape[1], x2 + pad)
        y2 = min(img.shape[0], y2 + pad)
        
        plate_crop = img[y1:y2, x1:x2]
        
        proc = preprocess_plate_for_paddle(plate_crop)
        raw_text = ocr_plate_paddle(proc)
        norm_text = normalize_plate_text(raw_text)
        
        if validate_indian_plate(norm_text):
            label = f"Plate: {norm_text}"
            color = (0,255,0)
        else:
            label = f"Invalid: {raw_text}"
            color = (0,0,255)
        
        cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
        cv2.putText(img, label, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

cv2.imshow("PaddleOCR Image Result", img)
cv2.waitKey(0)
cv2.destroyAllWindows()



0: 640x640 1 license_plate, 30.1ms
Speed: 3.6ms preprocess, 30.1ms inference, 2.5ms postprocess per image at shape (1, 3, 640, 640)
[2026/02/12 09:57:11] ppocr DEBUG: dt_boxes num : 1, elapse : 0.02321600914001465
[2026/02/12 09:57:11] ppocr DEBUG: cls num  : 1, elapse : 0.017665386199951172
[2026/02/12 09:57:11] ppocr DEBUG: rec_res num  : 1, elapse : 0.1298847198486328


## Video + Temporal Voting

In [ ]:
video_path = r"C:\Users\100ra\Downloads\How_High_Security_Number_Plate_challan_system_identifies_car_for_making_challan_digitalautomobile_720P.mp4"
cap = cv2.VideoCapture(video_path)

plate_buffer = deque(maxlen=7)
final_plate = None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    results = model(frame, conf=0.4)
    
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            
            pad = 5
            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)
            x2 = min(frame.shape[1], x2 + pad)
            y2 = min(frame.shape[0], y2 + pad)
            
            plate_crop = frame[y1:y2, x1:x2]
            
            proc = preprocess_plate_for_paddle(plate_crop)
            raw_text = ocr_plate_paddle(proc)
            norm_text = normalize_plate_text(raw_text)
            
            if validate_indian_plate(norm_text):
                plate_buffer.append(norm_text)
            
            if len(plate_buffer) >= 5:
                final_plate = max(set(plate_buffer), key=plate_buffer.count)
            
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            
            if final_plate:
                cv2.putText(frame, final_plate,
                            (x1, y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.8,
                            (0,255,0),
                            2)
    
    cv2.imshow("PaddleOCR ANPR", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 640x384 1 license_plate, 112.6ms
Speed: 11.5ms preprocess, 112.6ms inference, 3.1ms postprocess per image at shape (1, 3, 640, 384)
[2026/02/12 09:57:34] ppocr DEBUG: dt_boxes num : 1, elapse : 0.09990239143371582
[2026/02/12 09:57:34] ppocr DEBUG: cls num  : 1, elapse : 0.02161407470703125
[2026/02/12 09:57:34] ppocr DEBUG: rec_res num  : 1, elapse : 0.12620258331298828

0: 640x384 1 license_plate, 15.6ms
Speed: 2.2ms preprocess, 15.6ms inference, 2.3ms postprocess per image at shape (1, 3, 640, 384)
[2026/02/12 09:57:34] ppocr DEBUG: dt_boxes num : 1, elapse : 0.1038672924041748
[2026/02/12 09:57:34] ppocr DEBUG: cls num  : 1, elapse : 0.01808643341064453
[2026/02/12 09:57:34] ppocr DEBUG: rec_res num  : 1, elapse : 0.1282966136932373

0: 640x384 1 license_plate, 19.2ms
Speed: 2.2ms preprocess, 19.2ms inference, 4.1ms postprocess per image at shape (1, 3, 640, 384)
[2026/02/12 09:57:34] ppocr DEBUG: dt_boxes num : 1, elapse : 0.09393596649169922
[2026/02/12 09:57:34] ppocr DEBUG: